# Walmart Forecasting - Data Cleaning

This notebook loads the raw Walmart sales data, validates its quality, applies the core cleaning steps, and saves a cleaned dataset for EDA and forecasting.

In [13]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

DATA_PATH = Path("../data/Walmart.csv")
CLEANED_PATH = Path("../data/Walmart_cleaned.csv")

In [14]:
# Load the raw dataset
df = pd.read_csv(DATA_PATH)
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


In [15]:
# Quick structure check
print(f"Shape: {df.shape}")
display(df.info())
display(df.isna().sum())
display(df.dtypes)

Shape: (6435, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         6435 non-null   int64  
 1   Date          6435 non-null   object 
 2   Weekly_Sales  6435 non-null   float64
 3   Holiday_Flag  6435 non-null   int64  
 4   Temperature   6435 non-null   float64
 5   Fuel_Price    6435 non-null   float64
 6   CPI           6435 non-null   float64
 7   Unemployment  6435 non-null   float64
dtypes: float64(5), int64(2), object(1)
memory usage: 402.3+ KB


None

Store           0
Date            0
Weekly_Sales    0
Holiday_Flag    0
Temperature     0
Fuel_Price      0
CPI             0
Unemployment    0
dtype: int64

Store             int64
Date             object
Weekly_Sales    float64
Holiday_Flag      int64
Temperature     float64
Fuel_Price      float64
CPI             float64
Unemployment    float64
dtype: object

In [16]:
def audit_dataset(dataframe: pd.DataFrame, name: str) -> pd.DataFrame:
    summary = pd.DataFrame(
        {
            "dtype": dataframe.dtypes.astype(str),
            "missing_values": dataframe.isna().sum(),
            "missing_pct": (dataframe.isna().mean() * 100).round(2),
            "unique_values": dataframe.nunique()
        }
    )

    print(f"Dataset: {name}")
    print(f"Rows: {len(dataframe)}")
    print(f"Columns: {dataframe.shape[1]}")
    print(f"Duplicate rows: {dataframe.duplicated().sum()}")
    print(f"Duplicate Store-Date pairs: {dataframe.duplicated(subset=['Store', 'Date']).sum()}")
    print("\nColumn audit:")
    return summary


audit_dataset(df, "Raw Walmart Sales Data")

Dataset: Raw Walmart Sales Data
Rows: 6435
Columns: 8
Duplicate rows: 0
Duplicate Store-Date pairs: 0

Column audit:


,dtype,missing_values,missing_pct,unique_values
Store,int64,0,0.0,45
Date,object,0,0.0,143
Weekly_Sales,float64,0,0.0,6435
Holiday_Flag,int64,0,0.0,2
Temperature,float64,0,0.0,3528
Fuel_Price,float64,0,0.0,892
CPI,float64,0,0.0,2145
Unemployment,float64,0,0.0,349


## Cleaning Steps

The raw file has no missing values, but it still needs a few important cleaning steps:

- Parse the `Date` column correctly using day-first format.
- Check for duplicates and invalid values.
- Sort records in time order for each store.
- Add basic calendar columns that help later analysis.
- Save a cleaned version of the dataset.

In [17]:
# Work on a copy so the raw dataframe stays unchanged
clean_df = df.copy()

# Parse dates using the format in the CSV: dd-mm-yyyy
clean_df["Date"] = pd.to_datetime(clean_df["Date"], format="%d-%m-%Y", errors="coerce")

if clean_df["Date"].isna().any():
    raise ValueError("Some dates could not be parsed. Please inspect the raw data.")

# Remove exact duplicate rows if any appear in future updates
clean_df = clean_df.drop_duplicates().copy()

# Sort by store and date for time-series work
clean_df = clean_df.sort_values(["Store", "Date"]).reset_index(drop=True)

# Keep Holiday_Flag numeric, but ensure the values are valid
valid_holiday_values = {0, 1}
actual_holiday_values = set(clean_df["Holiday_Flag"].dropna().unique())
if not actual_holiday_values.issubset(valid_holiday_values):
    raise ValueError(f"Unexpected Holiday_Flag values found: {actual_holiday_values}")

# Add useful date parts for downstream EDA and forecasting
iso_calendar = clean_df["Date"].dt.isocalendar()
clean_df["Year"] = clean_df["Date"].dt.year
clean_df["Month"] = clean_df["Date"].dt.month
clean_df["Week"] = iso_calendar.week.astype(int)
clean_df["Quarter"] = clean_df["Date"].dt.quarter
clean_df["Is_Weekend"] = (clean_df["Date"].dt.dayofweek >= 5).astype(int)

clean_df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Week,Quarter,Is_Weekend
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,2010,2,5,1,0
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106,2010,2,6,1,0
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106,2010,2,7,1,0
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106,2010,2,8,1,0
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106,2010,3,9,1,0


In [18]:
# Business-rule validation checks
validation_summary = {
    "missing_values": int(clean_df.isna().sum().sum()),
    "duplicate_rows": int(clean_df.duplicated().sum()),
    "duplicate_store_date_pairs": int(clean_df.duplicated(subset=["Store", "Date"]).sum()),
    "negative_weekly_sales": int((clean_df["Weekly_Sales"] < 0).sum()),
    "invalid_holiday_flag": int((~clean_df["Holiday_Flag"].isin([0, 1])).sum()),
    "non_positive_fuel_price": int((clean_df["Fuel_Price"] <= 0).sum()),
    "non_positive_cpi": int((clean_df["CPI"] <= 0).sum()),
    "negative_unemployment": int((clean_df["Unemployment"] < 0).sum())
}

pd.Series(validation_summary, name="count")

missing_values                0
duplicate_rows                0
duplicate_store_date_pairs    0
negative_weekly_sales         0
invalid_holiday_flag          0
non_positive_fuel_price       0
non_positive_cpi              0
negative_unemployment         0
Name: count, dtype: int64

In [19]:
# Outlier summary for review. We do not remove outliers automatically in a forecasting project.
numeric_cols = ["Weekly_Sales", "Temperature", "Fuel_Price", "CPI", "Unemployment"]
outlier_summary = {}

for col in numeric_cols:
    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_summary[col] = int(((clean_df[col] < lower_bound) | (clean_df[col] > upper_bound)).sum())

pd.Series(outlier_summary, name="possible_outliers")

Weekly_Sales     34
Temperature       3
Fuel_Price        0
CPI               0
Unemployment    481
Name: possible_outliers, dtype: int64

In [20]:
# Final audit after cleaning
audit_dataset(clean_df, "Cleaned Walmart Sales Data")

Dataset: Cleaned Walmart Sales Data
Rows: 6435
Columns: 13
Duplicate rows: 0
Duplicate Store-Date pairs: 0

Column audit:


,dtype,missing_values,missing_pct,unique_values
Store,int64,0,0.0,45
Date,datetime64[ns],0,0.0,143
Weekly_Sales,float64,0,0.0,6435
Holiday_Flag,int64,0,0.0,2
Temperature,float64,0,0.0,3528
Fuel_Price,float64,0,0.0,892
CPI,float64,0,0.0,2145
Unemployment,float64,0,0.0,349
Year,int32,0,0.0,3
Month,int32,0,0.0,12


In [21]:
# Save cleaned data for later notebooks
clean_df.to_csv(CLEANED_PATH, index=False)
print(f"Cleaned dataset saved to: {CLEANED_PATH.resolve()}")
print(f"Final shape: {clean_df.shape}")

Cleaned dataset saved to: /Users/bharathkumarvnaik/Downloads/programing/Data_Scientist_projects/Walmart_Forecasting/data/Walmart_cleaned.csv
Final shape: (6435, 13)
